# Language Tracking — ITPC Analysis

Measures neural entrainment to speech rhythm via Inter-Trial Phase Coherence (ITPC).  
A healthy brain synchronises its phase to the temporal structure of connected speech at:

| Frequency | Rate | Linguistic level |
| --- | --- | --- |
| **0.78 Hz** | ~1.28 s / sentence | Sentence rate |
| **1.56 Hz** | ~0.64 s | Phrase rate (2× sentence) |
| **3.125 Hz** | ~0.32 s | Word rate |

A **positive finding** is a significant ITPC peak at one or more of these frequencies (p < 0.05 by permutation test), maximal over temporal/fronto-temporal channels.

**Environment:** `stimulus_software/.venv` (MNE 1.11, scipy, pandas, matplotlib)

## 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

SUBJECT_ID   = 'CON013'
SESSION_DATE = '2026-04-10'

ANALYSIS_ROOT = Path.cwd().resolve()
if ANALYSIS_ROOT.name != 'analysis':
    ANALYSIS_ROOT = next(
        (parent for parent in [ANALYSIS_ROOT, *ANALYSIS_ROOT.parents]
         if parent.name == 'analysis' and (parent / 'notebooks').exists()),
        ANALYSIS_ROOT,
    )
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from lib.io import DEFAULT_EEG_CHANNELS, build_subject_paths

paths = build_subject_paths(SUBJECT_ID, SESSION_DATE, analysis_root=ANALYSIS_ROOT)
EDF_PATH = paths['edf_path']
CSV_PATH = paths['csv_path']
OUT_DIR = str(paths['out_dir'])
EEG_CHANNELS = DEFAULT_EEG_CHANNELS.copy()
BAD_CHANNELS = []  # e.g. ['Fp1', 'T3']

print(f'Subject: {SUBJECT_ID}  |  EDF: {EDF_PATH}')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy.fft import fft
from scipy import stats

from lib.io import align_stimulus_csv, load_raw_eeg_metadata
from lib.preprocessing import load_filtered_eeg

%matplotlib inline
mne.set_log_level('WARNING')
print(f'MNE {mne.__version__}')

## 3. Load EDF + Sync Alignment

In [ ]:
raw, sfreq, available_eeg = load_raw_eeg_metadata(
    EDF_PATH,
    eeg_channels=EEG_CHANNELS,
    bad_channels=BAD_CHANNELS,
    preload=False,
    verbose=False,
)
print(f'EEG channels ({len(available_eeg)}): {available_eeg}')
print(f'sfreq: {sfreq} Hz  |  duration: {raw.n_times/sfreq:.0f}s')

In [ ]:
df, time_offset = align_stimulus_csv(CSV_PATH, sfreq=sfreq, n_times=raw.n_times)
print(f'time_offset = {time_offset:.4f} s')

## 4. Preprocessing

In [ ]:
# Narrow band-pass for ITPC (speech frequencies are below 5 Hz)
raw_lang = load_filtered_eeg(raw, available_eeg, l_freq=0.1, h_freq=25, verbose=False)
print('Preprocessing done.')

## 5. Epoch

In [ ]:
lang_df = df[df['stim_type'] == 'language'].copy()
print(f'Language trials: {len(lang_df)}')
dur = lang_df['edf_end'] - lang_df['edf_start']
print(f'Duration: mean={dur.mean():.2f}s  min={dur.min():.2f}s  max={dur.max():.2f}s')

lang_events = np.column_stack([
    lang_df['start_sample'].values,
    np.zeros(len(lang_df), dtype=int),
    np.ones(len(lang_df), dtype=int)
])

# 15.36s = 12 sentences × 1.28s
epochs = mne.Epochs(raw_lang, events=lang_events, event_id={'language': 1},
                    tmin=0, tmax=15.36, baseline=None, preload=True, verbose=False)
epochs.resample(256, verbose=False)

print(f'Epochs: {len(epochs)} trials × {len(epochs.ch_names)} ch × {epochs.get_data().shape[2]} samples')

## 6. Compute ITPC

In [ ]:
def compute_itpc(epochs_data, fs):
    """Returns itpc (n_samples, n_channels) and freq axis."""
    data = np.transpose(epochs_data, (2, 1, 0))  # (n_samples, n_ch, n_trials)
    freqs = np.fft.fftfreq(data.shape[0], 1 / fs)
    itpc = np.abs(np.exp(1j * np.angle(fft(data, axis=0))).mean(axis=2))
    return itpc, freqs

print(f'Computing ITPC on {len(epochs)} trials...')
itpc, freqs = compute_itpc(epochs.get_data(), fs=epochs.info['sfreq'])
print('Done.')

## 7. Permutation Test

Shuffle trial labels 1000× to build a null distribution. Compare observed ITPC at each target frequency to the null.

In [ ]:
N_PERMS = 1000
TARGET_FREQS = [0.78, 1.56, 3.125]

epochs_data = epochs.get_data()  # (n_trials, n_ch, n_samples)
fs = epochs.info['sfreq']

# Observed: avg ITPC across channels at each target freq
observed = {}
for f in TARGET_FREQS:
    idx = np.argmin(np.abs(freqs - f))
    observed[f] = itpc[idx, :].mean()

# Null distribution: shuffle phase across trials
print(f'Running {N_PERMS} permutations...')
null = {f: [] for f in TARGET_FREQS}
rng = np.random.default_rng(42)

for _ in range(N_PERMS):
    shuffled = epochs_data[rng.permutation(len(epochs_data))]
    perm_itpc, _ = compute_itpc(shuffled, fs)
    for f in TARGET_FREQS:
        idx = np.argmin(np.abs(freqs - f))
        null[f].append(perm_itpc[idx, :].mean())

print('\nResults:')
print(f'{"Frequency":>12}  {"Observed":>10}  {"Null mean":>10}  {"p-value":>10}  {"Significant":>12}')
results = {}
for f in TARGET_FREQS:
    obs = observed[f]
    null_arr = np.array(null[f])
    p = np.mean(null_arr >= obs)
    sig = '✓' if p < 0.05 else ''
    results[f] = {'observed': obs, 'null_mean': null_arr.mean(), 'p_value': p}
    print(f'{f:>12.3f}  {obs:>10.4f}  {null_arr.mean():>10.4f}  {p:>10.4f}  {sig:>12}')

## 8. Plots

In [ ]:
# Average ITPC spectrum with target bands and null distribution envelope
fmin, fmax = 0.5, 4.0
pos_idx = (freqs >= fmin) & (freqs <= fmax)
plot_freqs = freqs[pos_idx]
avg_itpc = itpc[pos_idx, :].mean(axis=1)

TARGETS = [(0.78, 'teal', '0.78 Hz'), (1.56, 'darkorchid', '1.56 Hz'), (3.125, 'firebrick', '3.125 Hz')]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(plot_freqs, avg_itpc, color='steelblue', lw=1.5, label='Observed ITPC')

# Plot null 95th percentile
null_95 = {}
for f, _, _ in TARGETS:
    null_95[f] = np.percentile(null[f], 95)

for f, c, lbl in TARGETS:
    p = results[f]['p_value']
    marker = ' *' if p < 0.05 else ''
    ax.axvspan(f - 0.04, f + 0.04, color=c, alpha=0.2, label=f'{lbl}{marker} (p={p:.3f})')
    ax.axhline(null_95[f], color=c, lw=0.8, ls=':', alpha=0.6)

ax.set(xlabel='Frequency (Hz)', ylabel='ITPC',
       title=f'{SUBJECT_ID} — Language ITPC (avg across {len(available_eeg)} channels)',
       xlim=(fmin, fmax))
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_lang_itpc_avg.png'), dpi=150)
plt.show()

In [ ]:
# Per-channel grid
n_ch = len(epochs.ch_names)
ncols = 4
nrows = (n_ch + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.5 * nrows), sharex=True, sharey=True)
for i, ch_name in enumerate(epochs.ch_names):
    ax = axes.flatten()[i]
    ax.plot(plot_freqs, itpc[pos_idx, i], color='steelblue', lw=1)
    for f, c, _ in TARGETS:
        ax.axvspan(f - 0.04, f + 0.04, color=c, alpha=0.2)
    ax.set_title(ch_name, fontsize=9)
    ax.grid(True, alpha=0.2)
for j in range(i + 1, len(axes.flatten())):
    axes.flatten()[j].set_visible(False)
fig.suptitle(f'{SUBJECT_ID} — Language ITPC per channel', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_lang_itpc_channels.png'), dpi=120)
plt.show()

In [ ]:
# Topomap at each target frequency
montage = mne.channels.make_standard_montage('standard_1020')
epochs.set_montage(montage, match_case=False, on_missing='warn')

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 4))
for ax, (f, c, lbl) in zip(axes, TARGETS):
    idx = np.argmin(np.abs(freqs - f))
    ch_itpc = itpc[idx, :]
    im, _ = mne.viz.plot_topomap(ch_itpc, epochs.info, axes=ax, show=False,
                                  vlim=(0, ch_itpc.max()), cmap='hot_r')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    p = results[f]['p_value']
    ax.set_title(f'{lbl}  (p={p:.3f})', fontsize=10)
fig.suptitle(f'{SUBJECT_ID} — Language ITPC topomap', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'{SUBJECT_ID}_lang_itpc_topomap.png'), dpi=150)
plt.show()

## 9. Summary

**Positive finding:** ITPC peak at 0.78 Hz and/or 1.56 Hz, p < 0.05, maximal over temporal channels (T3/T4/T5/T6).  
**Clinical interpretation:** Preserved covert language comprehension.